In [521]:
%mkdir data

In [525]:
%cd data

C:\Users\AMD\Desktop\sarvam\data


In [527]:
!pip install wget
!python -m wget https://dl.fbaipublicfiles.com/fasttext/vectors-wiki/wiki.hi.vec


Saved under wiki.hi.vec


In [529]:
!python -m wget https://dl.fbaipublicfiles.com/fasttext/vectors-wiki/wiki.en.vec


Saved under wiki.en.vec


In [531]:
!python -m wget https://dl.fbaipublicfiles.com/arrival/dictionaries/en-hi.txt
!python -m wget https://dl.fbaipublicfiles.com/arrival/dictionaries/en-hi.5000-6500.txt


Saved under en-hi.txt

Saved under en-hi.5000-6500.txt


In [533]:
%cd ..

C:\Users\AMD\Desktop\sarvam


#### Preparation Stage:

Preparation stage (load_embeddings) uses the pre-trained FastText embeddings for Hiddi and English available from the fasttext.cc .

The vocab limit is used to limit the frequency of words being used. In pre-trained FastText files, words are typically sorted by frequency (with the most frequent words first)[1], so loading the first 100,000 lines gives the top vocabulary.

load_bilingual_lexicon loads the bilingual lexicon from the MUSE dataset of en-hi.
##### References:

[1] https://fasttext.cc/docs/en/unsupervised-tutorial.html

In [535]:
import io
import numpy as np
import torch
import os

def load_embeddings(file_path, vocab_limit=100000, emb_dim=None, full_vocab=False, cuda=False):

    word2id = {}
    vectors = []
    if not os.path.isabs(file_path):
        path = os.path.abspath(os.path.join(os.getcwd(), file_path))
    else:
        path = file_path

    print("Loading file from:", path)
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found at: {path}")

    with io.open(path, 'r', encoding='utf-8', newline='\n', errors='ignore') as f:
        for i, line in enumerate(f):
            if i == 0:
                split = line.split()
                if len(split) == 2 and split[0].isdigit() and split[1].isdigit():
                    if emb_dim is None:
                        emb_dim = int(split[1])
                    else:
                        assert emb_dim == int(split[1]), (
                            f"Expected emb_dim={emb_dim}, but found {split[1]} in header."
                        )
                    continue
            parts = line.rstrip().split(' ', 1)
            if len(parts) != 2:
                continue
            word, vect_str = parts

            if not full_vocab:
                word = word.lower()

            vect = np.fromstring(vect_str, sep=' ')

            if np.linalg.norm(vect) == 0:
                vect[0] = 0.01

            if emb_dim is not None and vect.shape[0] != emb_dim:
                continue

            if word in word2id:
                continue

            word2id[word] = len(word2id)
            vectors.append(vect[None])

            if vocab_limit > 0 and len(word2id) >= vocab_limit and not full_vocab:
                break
    if not vectors:
        raise ValueError("No embeddings were loaded. Check the file format and path.")

    emb_array = np.concatenate(vectors, axis=0)

    embeddings = torch.from_numpy(emb_array).float()
    if cuda:
        embeddings = embeddings.cuda()

    print(f"Loaded {len(word2id)} word embeddings from {file_path}.")
    return word2id, embeddings



In [537]:
word2id_hindi, embeddings_hindi = load_embeddings("./data/wiki.hi.vec", vocab_limit=100000, emb_dim=300, full_vocab=False, cuda=True)
word2id_english, embeddings_english = load_embeddings("./data/wiki.en.vec", vocab_limit=100000, emb_dim=300, full_vocab=False, cuda=True)
sample_word_hindi = "के"

if sample_word_hindi in word2id_hindi:
    idx = word2id_hindi[sample_word_hindi]
    sample_vector = embeddings_hindi[idx].cpu().numpy() if embeddings_hindi.is_cuda else embeddings_hindi[idx].numpy()
    print(f"Sample vector for '{sample_word_hindi}':\n{sample_vector}")
else:
    print(f"Word '{sample_word_hindi}' not found in the embeddings.")

sample_word_english = "the"

if sample_word_english in word2id_english:
    idx = word2id_english[sample_word_english]
    sample_vector = embeddings_english[idx].cpu().numpy() if embeddings_english.is_cuda else embeddings_english[idx].numpy()
    print(f"Sample vector for '{sample_word_english}':\n{sample_vector}")
else:
    print(f"Word '{sample_word_english}' not found in the embeddings.")


Loading file from: C:\Users\AMD\Desktop\sarvam\data\wiki.hi.vec
Loaded 100000 word embeddings from ./data/wiki.hi.vec.
Loading file from: C:\Users\AMD\Desktop\sarvam\data\wiki.en.vec
Loaded 100000 word embeddings from ./data/wiki.en.vec.
Sample vector for 'के':
[ 0.046271   -0.062825   -0.13019    -0.081064    0.20132     0.035899
  0.054962   -0.021752    0.11351     0.0056929  -0.0027507  -0.0021876
 -0.35342    -0.021807    0.10967     0.028848   -0.0086524  -0.035754
  0.10435    -0.12547     0.10898     0.089893    0.31709     0.13962
  0.1024      0.0047501  -0.037492    0.062767    0.084519    0.20156
  0.049685    0.081132   -0.061762    0.16084     0.052234   -0.063994
  0.14446     0.02963     0.23153    -0.059726   -0.031571   -0.088169
 -0.23371    -0.10874    -0.12587    -0.0064134  -0.041136   -0.11884
  0.1608     -0.062858   -0.052549    0.14045     0.13036     0.16015
  0.14402    -0.27292    -0.22844     0.036744   -0.13193     0.16101
  0.0097727   0.031117    0.1373

In [539]:
def load_bilingual_lexicon(file_path):

    source_words = []
    target_words = []
    if not os.path.isabs(file_path):
        path = os.path.abspath(os.path.join(os.getcwd(), file_path))
    else:
        path = file_path

    print("Loading file from:", path)
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found at: {path}")

    with open(file_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            tokens = line.split()
            if len(tokens) < 2:
                continue

            source, target = tokens[0], tokens[1]
            source_words.append(source)
            target_words.append(target)


    print(f"Loaded {len(source_words)} bilingual word pairs from {file_path}.")
    return source_words, target_words


In [541]:
def create_dico(source_words,target_words, cuda=False):

        pairs = []
        for s_word, t_word in zip(source_words, target_words):
            if s_word in word2id_english and t_word in word2id_hindi:
                pairs.append([word2id_english[s_word], word2id_hindi[t_word]])
        if not pairs:
            raise ValueError("No matching bilingual pairs found between lexicon and embeddings.")
        dico_tensor = torch.tensor(pairs, dtype=torch.long)
        if cuda:
            dico_tensor = dico_tensor.cuda()
        return dico_tensor

#### Embedding Alignment using Procrustes method
Idea is to learn an orthgonal mapping W such that when applied to source A (ENGLISH) embeddings, the mapped vectors WA aligns closely with the target B (hindi) embeddings.

orthogonalize function alignment using the formula present in the paper by Conneau et al [1]. Paper also mentions that β = 0.01 performs by ensuring the matrix stays close to the orthogonal matrices after each update.
W ← (1 + β)W − β(WWT
)

check_ortho is used to see if the W transpose multiplied with W leads to the identity matrix which ensures there is orthogonality. 


##### References
1) Conneau, A., Lample, G., Ranzato, M. A., Denoyer, L., & Jégou, H. (2017). Word Translation Without Parallel Data. CoRR, abs/1710.04087. Retrieved from http://arxiv.org/abs/1710.04087W

In [569]:
mapping = torch.nn.Linear(300, 300, bias=False)
if torch.cuda.is_available():
          mapping = mapping.cuda()
import scipy
def procrustes(embeddings_english, embeddings_hindi, dico):

        A = embeddings_english[dico[:, 0]]
        B = embeddings_hindi[dico[:, 1]]
        W = mapping.weight.data

        M = B.transpose(0, 1).mm(A)
        if W.is_cuda:
            M_cpu = M.cpu().numpy()
        else:
            M_cpu = M.numpy()

        U, S, V_t = scipy.linalg.svd(M_cpu, full_matrices=True)
        optimal_W = U.dot(V_t)
        optimal_W_tensor = torch.from_numpy(optimal_W).type_as(W)
        W.copy_(optimal_W_tensor)
        print("Procrustes mapping updated.")


In [571]:
def orthogonalize():
        W = mapping.weight.data
        beta = 0.01  
        W.copy_((1 + beta) * W - beta * W.mm(W.transpose(0, 1).mm(W)))
        logger.info("Mapping matrix orthogonalized with beta=0.01")

In [573]:
def check_ortho():
    W = mapping.weight.data
    W_t = torch.mm(W.transpose(0, 1), W)
    identity = torch.eye(W_t.size(0), dtype=W_t.dtype, device=W_t.device)
    if torch.allclose(W_t, identity, atol=1e-5):
        print("The mapping is orthogonal: W^T * W is approximately the identity matrix.")
    else:
        deviation = torch.max(torch.abs(W_t - identity))
        print("The mapping is not orthogonal. Maximum deviation from identity: {:.6f}".format(deviation))



#### Evaluation

Map English word vectors through the computed transformation 𝑊.

Retrieve the corresponding Hindi words by finding the nearest neighbors using a cosine similarity measure.

Precision@1: The proportion of correct translations in the top-1 predictions.

Precision@5: The proportion where the correct translation appears within the top 5 predictions.


In [589]:
import os
import logging
logger = logging.getLogger(__name__)


def load_dictionary(file_path, word2id_english, word2id_hindi):


    if not os.path.isabs(file_path):
        path = os.path.abspath(os.path.join(os.getcwd(), file_path))
    else:
        path = file_path

    print("Loading file from:", path)
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found at: {path}")

    pairs = []
    not_found = 0
    not_found1 = 0
    not_found2 = 0

    with io.open(path, 'r', encoding='utf-8') as f:
        for index, line in enumerate(f):
            # Expect dictionary file to be in lowercase.
            assert line == line.lower(), f"Line not in lowercase: {line}"
            parts = line.rstrip().split()
            if len(parts) < 2:
                logger.warning("Could not parse line %s (%i)", line, index)
                continue
            word1, word2 = parts
            if word1 in word2id_english and word2 in word2id_hindi:
                pairs.append((word1, word2))
            else:
                not_found += 1
                not_found1 += int(word1 not in word2id_english)
                not_found2 += int(word2 not in word2id_hindi)

    logger.info("Found %i pairs of words in the dictionary (%i unique). "
                "%i other pairs contained at least one unknown word "
                "(%i in lang1, %i in lang2)",
                len(pairs), len(set([x for x, _ in pairs])),
                not_found, not_found1, not_found2)

    pairs = sorted(pairs, key=lambda x: word2id_english[x[0]])
    dico = torch.LongTensor(len(pairs), 2)
    for i, (word1, word2) in enumerate(pairs):
        dico[i, 0] = word2id_english[word1]
        dico[i, 1] = word2id_hindi[word2]
    return dico

def evaluation(path):
        
        aligned_embeddings_english = mapping(embeddings_english).data
        aligned_embeddings_hindi = embeddings_hindi.data
        dico = load_dictionary(path, word2id_english, word2id_hindi)
        dico = dico.cuda() if aligned_embeddings_english.is_cuda else dico
        dico = dico.to(aligned_embeddings_english.device)
        assert dico[:, 0].max() < aligned_embeddings_english.size(0)
        assert dico[:, 1].max() < embeddings_hindi.size(0)

        emb1 = aligned_embeddings_english / aligned_embeddings_english.norm(p=2, dim=1, keepdim=True)
        emb2 = embeddings_hindi / embeddings_hindi.norm(p=2, dim=1, keepdim=True)
        
        query = emb1[dico[:, 0]]

        scores = query.mm(emb2.transpose(0, 1))

        top_matches = scores.topk(10, 1, True)[1]
        results = {}
        # precision@1, precision@5, precision@10 for the word translation using test data set
        for k in [1, 5,10]:
            top_k_matches = top_matches[:, :k]
            _matching = (top_k_matches == dico[:, 1][:, None].expand_as(top_k_matches)).sum(1).cpu().numpy()
            matching = {}
            for i, src_id in enumerate(dico[:, 0].cpu().numpy()):
                matching[src_id] = min(matching.get(src_id, 0) + _matching[i], 1)
            precision_at_k = 100 * np.mean(list(matching.values()))
            results[f'precision_at_{k}'] = precision_at_k
            print(f"Precision@{k}: {precision_at_k:.2f}")


        cosine_similarities = (emb1 * emb2).sum(dim=1)
        print("Average cosine similarity:", cosine_similarities.mean().item())



In [591]:
source_words, target_words = load_bilingual_lexicon("./data/en-hi.txt")
test_path = './data/en-hi.5000-6500.txt'
dico = create_dico(source_words, target_words, True)
def run_pipeline(dico):

    procrustes(embeddings_english,embeddings_hindi,dico)
    orthogonalize()
    check_ortho()
    evaluation(test_path)
    

for size in [5000,10000,15000, 20000, 35000]:
    print("\n dictionary size results for %s : \n", size)
    dico_res = dico[:size]
    run_pipeline(dico_res)

print("\n Full dictionary size results : \n")
run_pipeline(dico)    

Loading file from: C:\Users\AMD\Desktop\sarvam\data\en-hi.txt
Loaded 38221 bilingual word pairs from ./data/en-hi.txt.

 dictionary size results for %s : 
 5000
Procrustes mapping updated.
The mapping is orthogonal: W^T * W is approximately the identity matrix.
Loading file from: C:\Users\AMD\Desktop\sarvam\data\en-hi.5000-6500.txt
Precision@1: 23.56
Precision@5: 43.01
Precision@10: 51.16
Average cosine similarity: 0.17440687119960785

 dictionary size results for %s : 
 10000
Procrustes mapping updated.
The mapping is orthogonal: W^T * W is approximately the identity matrix.
Loading file from: C:\Users\AMD\Desktop\sarvam\data\en-hi.5000-6500.txt
Precision@1: 38.01
Precision@5: 56.44
Precision@10: 62.53
Average cosine similarity: 0.17872829735279083

 dictionary size results for %s : 
 15000
Procrustes mapping updated.
The mapping is orthogonal: W^T * W is approximately the identity matrix.
Loading file from: C:\Users\AMD\Desktop\sarvam\data\en-hi.5000-6500.txt
Precision@1: 38.84
Preci

#### Results and Disscussion

The best performance seems to be achieved using between 10000 and 15000 word pairs. The results indicate that there is a sweet spot where the lexicon contains sufficient high-quality translations to learn effective mapping without being diluted by noisier entries that might emerge in larger sets.

Across all experiments, every Procrustes update confirms that the mapping remains orthogonal (i.e., 𝑊𝑇.𝑊 ) is approximately the identity matrix). This is important as it ensures that the transformation preserves distances and the geometric structure of the embedding space.

The slight increases in average cosine similarity with larger lexicon sizes suggest that the geometric alignment (in terms of vector orientation and distances) becomes marginally better; however, this improvement in similarity is not sufficient to overcome the noise introduced by additional, potentially less reliable pairs in terms of translation accuracy.


